# FLAG 2027 — NB-45: FACELORA-02, a LoRA adapter on the face encoder farl16 trained against frozen SimAM voices

**Why.** Every gain so far came from heads on FROZEN encoders. The voice side may not be fine-tuned on Bengali speech, but
the **face encoder may**. This notebook pulls the face embedding of FaRL ep16 (openai CLIP ViT-B/16 + FaRL weights, as `farl16` of FLAG_40) towards the SimAM voice space with a LoRA adapter
(rank 16 on the MLPs `c_fc` / `c_proj` of blocks 6-11 of 12), trained on Bengali-filtered VoxCeleb2 only, then embeds every MAV-Celeb row
and the VoxCeleb2 frames with it (`farlA`, 512-d). Whether it helps is decided by the FACELORA-02 lab notebook (FLAG_59:
Track 3 recipe, arms B0 / FA / CA / FCA, same seeds), registered in `Experiment/FACELORA-02/README.md`.

**Registered (no selection by result):** P = 32 identities x K = 4 (face, voice) pairs, face from a video different from
the voice's, random horizontal flip; symmetric multi-positive InfoNCE, tau 0.05; AdamW (LoRA 1e-4, voice projection 1e-3,
wd 1e-2), 100 warm-up steps, cosine, **1,500 steps**, fp16, seed 0; gradient checkpointing on the adapted blocks (memory
only). 300 VoxCeleb2 identities are held out only to print a sanity EER (steps 0 / 500 / 1000 / 1500).
**Wiring check:** at step 0 (B = 0) the adapter IS the frozen encoder, so 64 VoxCeleb2 frames must reproduce FLAG_41's
frozen arrays (mean cosine >= 0.99; frames cached as lossless PNG), else the notebook stops before training.

| output | content |
|---|---|
| `face_farlA_*.npy`, `ext_*_face_farlA.npy` | adapted embedding of every MAV-Celeb row (FLAG_40 row order) |
| `vox_face_farlA.npy` (+ `face_meta.csv`) | the same for the 29,251 VoxCeleb2 frames, FLAG_37 / FLAG_41 row order (NaN = frame not fetched) |
| `facelora_state.pt`, `train_log.json`, `vproj_W.npy`, `vproj_b.npy` | adapter, loss / sanity-EER curve, voice projection |
| `vox_frames_png.zip` | cached frames, lossless PNG (resume; the other FACELORA notebook can reuse them) |

**Rules:** face encoder fine-tuning is allowed; the voice encoders stay frozen; training data = VoxCeleb2 with the
Bengali-flagged speakers of FLAG_42 removed (**required**: the notebook stops without `lid_summary.json`); no MAV-Celeb
identity is used for training or compared with VoxCeleb2. Declare in the system description.

**Inputs (Add Input):** raw v4 (`train_set` + `dev_set`), MAV-Celeb v1/v2 + `flag2027-ext-meta` (as FLAG_40);
**`flag2027-lab-data`** (FLAG_41's VoxCeleb2 arrays, FLAG_37's key lists, FLAG_42's `lid_summary.json`; or the outputs of
FLAG_41 / FLAG_42 themselves); optional: this notebook's previous output (resume). **Settings:** GPU T4, **Internet ON**.

**Time:** about 3-4.5 h: VoxCeleb2 frames ~1.5 h (29k HTTP range reads; skipped when a `vox_frames_png.zip` is attached), training
~25 min, MAV-Celeb faces ~30 min, VoxCeleb2 frames ~10-20 min. Resumable (frames, adapter, finished arrays).

**Output:** `/kaggle/working/feats_facelora/`.

In [ ]:
import os, sys, io, re, time, json, types, shutil, zipfile, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
LOCAL = os.environ.get("FLAG_LOCAL") == "1"          # CPU smoke test on a mini copy of the data
IN = Path(os.environ.get("FLAG_IN", "/kaggle/input"))
WORKDIR = Path(os.environ.get("FLAG_OUT", "/kaggle/working"))
CACHE = Path(os.environ.get("FLAG_CACHE", "/tmp/flag_cache"))           # checkpoints stay out of the output
ECAPA_DIR = Path(os.environ.get("FLAG_ECAPA", str(CACHE / "ecapa_sb")))
OUT = WORKDIR / "feats_facelora"
for d in (WORKDIR, CACHE, OUT):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)

# ------------------------------------------------------------------ what to run (edit here)
ENCODERS = ['farlAV']   # the adapted farl16 face encoder (trained below)
EXT_SOURCES = ["v1_complete", "v2_complete"]                        # MAV-Celeb rows for STRESS-01; [] = skip
EXT_ENCODERS = list(ENCODERS)                                       # every model also on the v1/v2 rows
SPLIT_FILTER = None            # e.g. ["no_gender/English"]; None = every split found (train + every test list)
ZIP_PREFIXES = ("train_set", "dev_set", "eval", "test")             # organiser zips unpacked when not extracted
ONLY_UNDER = None              # e.g. "eval": keep only lists whose path contains this (dev and eval both attached)
print("LOCAL" if LOCAL else "KAGGLE", "| input", IN, "| output", OUT, "| cache", CACHE)
SMOKE_ROWS = int(os.environ.get("FLAG_SMOKE_ROWS", "0"))   # local smoke test only: rows kept per split / source
if (LOCAL or os.environ.get("FLAG_SMOKE") == "1") and os.environ.get("FLAG_ENCODERS"):              # local smoke test only: a subset of the encoders
    ENCODERS = EXT_ENCODERS = os.environ["FLAG_ENCODERS"].split(",")
if (LOCAL or os.environ.get("FLAG_SMOKE") == "1") and os.environ.get("FLAG_SPLITS"):                # local smoke test only: a subset of the organiser lists
    SPLIT_FILTER = os.environ["FLAG_SPLITS"].split(",")
LORA_ENC, ARR = "farl16", "farlA"   # FACELORA-02 variant of this notebook

In [ ]:
# openai CLIP (commit d50d76d, FaRL's code base) --no-deps; av decodes the face tracks.
if not LOCAL:
    !pip -q install ftfy regex av 2>&1 | tail -1
    !pip -q install --no-deps "git+https://github.com/openai/CLIP.git@d50d76daa670286dd6cacf3bcd80b5e4823fc8e1" 2>&1 | tail -1
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, soundfile as sf, transformers
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device", DEV,
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

## 1. Find the data (FLAG_12 cell): organiser lists with media, v1/v2 rows, resume

In [ ]:
# ============================================================================ finding the data
def scan_inputs(inp: Path, media=("faces", "voices")):
    """One os.walk over the inputs that never enters a faces/ or voices/ folder (an extracted MAV-Celeb holds hundreds of
    thousands of files; every rglob over it costs minutes). The organiser csv live in features/{faces,voices}/: entered.
    -> {"files": name -> [paths], "roots": folders holding both faces/ and voices/}"""
    files, roots = {}, []
    for dp, dns, fns in os.walk(inp, followlinks=True):
        feat = Path(dp).name == "features"
        if not feat and all(m in dns for m in media):
            roots.append(Path(dp))
        dns[:] = sorted(d for d in dns if feat or d not in media)
        for f in fns:
            files.setdefault(f, []).append(Path(dp) / f)
    log(f"scanned {inp}: {sum(map(len, files.values()))} files outside media folders, {len(roots)} media roots")
    return dict(files=files, roots=roots)


def media_cols(txt: Path):
    """Column of the .wav and of the image in an organiser list (train: 2 / 3; test: 1 / 2), or None."""
    try:
        row = pd.read_csv(txt, sep=" ", header=None, nrows=1).iloc[0]
    except Exception:
        return None
    wav = [i for i, v in enumerate(row) if str(v).lower().endswith(".wav")]
    jpg = [i for i, v in enumerate(row) if str(v).lower().endswith((".jpg", ".jpeg", ".png"))]
    if not wav or not jpg:
        return None
    return wav[0], jpg[0]


def has_media(txt: Path, cols):
    row = pd.read_csv(txt, sep=" ", header=None, nrows=1).iloc[0]
    return (txt.parent / row[cols[0]]).exists() and (txt.parent / row[cols[1]]).exists()


def split_name(txt: Path):
    """train_English.txt -> 'train'; <folder>/English_test.txt -> '<folder>/English' (dev: 'no_gender/English')."""
    if txt.name == "train_English.txt":
        return "train"
    return f"{txt.parent.name}/{re.sub(r'_test$', '', txt.stem)}"


tag = lambda split: split.replace("/", "_")


def find_splits(scan):
    """Every organiser list whose media sit beside it. A features-only copy of a list (no media) is ignored.
    Two different lists with the same split name (dev and Evaluation both attached) stop the run: set ONLY_UNDER."""
    S = {}
    for name, paths in scan["files"].items():
        if not name.endswith(".txt") or not (name == "train_English.txt" or name.endswith("_test.txt")):
            continue
        for p in sorted(paths, key=lambda q: len(q.as_posix())):
            if ONLY_UNDER and ONLY_UNDER not in p.as_posix() and name != "train_English.txt":
                continue
            cols = media_cols(p)
            if cols is None or not has_media(p, cols):
                continue
            k = split_name(p)
            df = pd.read_csv(p, sep=" ", header=None)
            if k in S:
                same = len(df) == len(S[k]["wav"]) and df[cols[0]].tolist() == S[k]["wav"]
                assert same, f"two different lists named {k}: {S[k]['txt']} and {p} -> set ONLY_UNDER"
                continue
            S[k] = dict(txt=p, root=p.parent, wav=df[cols[0]].tolist(), jpg=df[cols[1]].tolist())
    return S


def unpack_zips(scan, work: Path):
    zips = [p for n, ps in scan["files"].items() if n.lower().endswith(".zip") and n.lower().startswith(ZIP_PREFIXES)
            for p in ps]
    for zp in zips:
        marker = work / f".{zp.stem}.done"
        if not marker.exists():
            log("unpacking", zp)
            with zipfile.ZipFile(zp) as z:
                z.extractall(work)
            marker.touch()
    return bool(zips)


SCAN = scan_inputs(IN)
S = find_splits(SCAN)
if "train" not in S or len(S) == 1:                  # something may still be zipped (Kaggle keeps some uploads as .zip)
    if unpack_zips(SCAN, WORKDIR / "raw"):
        for k, v in find_splits(scan_inputs(WORKDIR / "raw")).items():
            S.setdefault(k, v)
if SPLIT_FILTER:
    S = {k: v for k, v in S.items() if k in SPLIT_FILTER}
assert S, "no organiser list with media found: attach the raw v4 data (or the Evaluation data)"
def wav_duration(p):
    """Seconds; 1.0 for a file soundfile cannot open (load_audio then returns 1 s of silence for it)."""
    try:
        return sf.info(str(p)).duration
    except Exception:
        return 1.0


for k, v in S.items():
    v["dur"] = np.array([wav_duration(v["root"] / p) for p in v["wav"]], dtype=np.float32)
    shutil.copy(v["txt"], OUT / f"index_{tag(k)}.txt")
    log(f"{k:22s} {len(v['wav']):5d} rows | voice median {np.median(v['dur']):.1f}s, max {v['dur'].max():.1f}s | {v['txt']}")
meta = SCAN["files"].get("meta_file_train_set.csv")
if meta:
    shutil.copy(meta[0], OUT / "meta_file_train_set.csv")


# ============================================================================ MAV-Celeb v1/v2 rows (optional)
def media_key(member):
    """'.../faces/<id>/<lang>/<video>/<file>' -> 'faces/<id>/<lang>/<video>/<file>' (drops any version folder)."""
    p = member.replace("\\", "/").split("/")
    for kind in ("faces", "voices"):
        if kind in p:
            return "/".join(p[p.index(kind):])
    return None


def ext_source(scan, name):
    """The attached zip for `name` (largest copy), else the extracted folder whose path names the source."""
    norm = lambda t: t.lower().replace("-", "_")
    zips = [p for n, ps in scan["files"].items() if n.lower().endswith(".zip") and norm(name) in norm(n) for p in ps]
    if zips:
        return max(zips, key=lambda p: p.stat().st_size)
    short = name.split("_")[0]
    for r in scan["roots"]:
        hint = norm(r.as_posix()) + "/"
        if f"/{short}/" in hint or name in hint:
            return r
    return None


class ExtRows:
    """The exact rows of ext_<name>_{face,voice}_meta.csv, read from the attached zip or folder (row order = csv)."""

    def __init__(self, src, name, meta_dir: Path, max_sec=12.0):
        fm = pd.read_csv(meta_dir / f"ext_{name}_face_meta.csv").path
        vm = pd.read_csv(meta_dir / f"ext_{name}_voice_meta.csv").path
        src = Path(src)
        if src.is_dir():
            from concurrent.futures import ThreadPoolExecutor
            self.face, self.voice = [src / media_key(p) for p in fm], [src / media_key(p) for p in vm]
            with ThreadPoolExecutor(32) as ex:
                ok = list(ex.map(lambda q: q.exists(), self.face + self.voice))
            if not all(ok):
                raise KeyError(f"{ok.count(False)} rows missing, e.g. {(self.face + self.voice)[ok.index(False)]}")
            self.read = lambda q: q.read_bytes()
        else:
            z = zipfile.ZipFile(src)
            by = {media_key(i.filename): i.filename for i in z.infolist() if not i.is_dir() and media_key(i.filename)}
            self.face, self.voice = [by[media_key(p)] for p in fm], [by[media_key(p)] for p in vm]
            self.read = z.read
        self.L = int(max_sec * 16000)          # 12 s centre crop, as the organiser-space features of these rows
        log(f"{name}: {len(self.face)} face rows, {len(self.voice)} voice rows in {src}")


EXT = {}
for name in EXT_SOURCES:
    metas = sorted(SCAN["files"].get(f"ext_{name}_voice_meta.csv", []), key=lambda p: -p.stat().st_size)
    src = ext_source(SCAN, name)
    if metas and src is not None:
        try:
            EXT[name] = ExtRows(src, name, metas[0].parent)
        except Exception as e:                    # optional: a bad zip or a missing row must not stop the run
            log(f"{name}: skipped ({type(e).__name__}: {e})")
    else:
        log(f"{name}: skipped (meta csv found: {bool(metas)}, source found: {src is not None})")


# ============================================================================ resume
def seed_from_previous(scan, out: Path):
    """Copy arrays of an earlier run of THIS notebook (a folder named like flag2027-feats-eval / feats_eval), so finished
    work is skipped. Arrays of other datasets (feats_v2, feats_models) are references for the check, never copied."""
    copied = []
    for n, ps in scan["files"].items():
        if not n.endswith(".npy") or (out / n).exists():
            continue
        for p in ps:
            if re.search(r"feats[-_]facelora", p.as_posix()) and out not in p.parents:
                shutil.copy(p, out / n); copied.append(n); break
    log(f"resume: {len(copied)} arrays copied from an earlier run")
    return copied


SEEDED = seed_from_previous(SCAN, OUT)
print({k: len(v["wav"]) for k, v in S.items()}, "| external:", list(EXT))
if (LOCAL or os.environ.get("FLAG_SMOKE") == "1") and SMOKE_ROWS:                     # local smoke test: the first rows of every split and source
    for v in S.values():
        v.update(wav=v["wav"][:SMOKE_ROWS], jpg=v["jpg"][:SMOKE_ROWS], dur=v["dur"][:SMOKE_ROWS])
    for r in EXT.values():
        r.face, r.voice = r.face[:SMOKE_ROWS], r.voice[:SMOKE_ROWS]
    log(f"smoke test: every split and source cut to {SMOKE_ROWS} rows")

## 2. Audio loading, batching, helpers (FLAG_12 cell)

In [ ]:
# ============================================================================ audio + batching
TARGET_SR = 16000
AUDIO_STATS = {"files": 0, "resampled": 0, "multichannel": 0, "short": 0}
MIN_SEC = 0.1                  # shorter clips are tiled to 0.1 s (w2v2 / AudioCLIP STFT fail on a few ms; v4 clips are >= 2 s)
BROKEN = {"audio": [], "image": []}   # unreadable sources: scored from a stand-in, listed in the manifest


def _src_name(src):
    """Full path of a file source (base names repeat across splits); the archive member name of a zip stream."""
    if isinstance(src, (bytes, bytearray)):
        return f"<{len(src)} bytes>"
    return str(src) if isinstance(src, (str, Path)) else str(getattr(src, "name", None) or src)


def load_audio(src):
    """Mono float32 at 16 kHz as the organiser loads audio (librosa.load(sr=16000, mono=True): channel mean + soxr_hq).
    All v4 wavs are 16 kHz mono; MAV-Celeb v1 has 44.1 kHz files. `src` = path or file-like.
    A file that cannot be read (or is empty / non-finite) -> 1 s of silence, listed in BROKEN["audio"]."""
    try:
        w, sr = sf.read(src if hasattr(src, "read") else str(src), dtype="float32", always_2d=True)
        if w.size == 0 or not np.isfinite(w).all():
            raise ValueError(f"{w.shape[0]} samples, finite {bool(np.isfinite(w).all())}")
    except Exception as e:
        BROKEN["audio"].append(f"{_src_name(src)} | {type(e).__name__}: {e}")
        log("!! unreadable audio -> 1 s of silence:", BROKEN["audio"][-1])
        return np.zeros(TARGET_SR, np.float32)
    AUDIO_STATS["files"] += 1
    if w.shape[1] > 1:
        AUDIO_STATS["multichannel"] += 1
        w = w.mean(1, keepdims=True)
    w = w[:, 0]
    if sr != TARGET_SR:
        import librosa
        w = librosa.resample(w, orig_sr=sr, target_sr=TARGET_SR).astype(np.float32)
        AUDIO_STATS["resampled"] += 1
    if len(w) < int(MIN_SEC * TARGET_SR):
        AUDIO_STATS["short"] += 1
        w = np.tile(w, int(np.ceil(MIN_SEC * TARGET_SR / len(w))))[:int(MIN_SEC * TARGET_SR)]
    return w


def centre_crop(w, L):
    if len(w) <= L:
        return w
    s = (len(w) - L) // 2
    return w[s:s + L]


def pack_batches(durs, max_padded_sec=240.0, max_bs=32):
    """Longest first; a batch costs len(batch) * longest item after padding."""
    order = np.argsort(-np.asarray(durs), kind="stable")
    batches, cur = [], []
    for i in order:
        longest = durs[cur[0]] if cur else durs[i]
        if cur and ((len(cur) + 1) * longest > max_padded_sec or len(cur) >= max_bs):
            batches.append(cur); cur = []
        cur.append(int(i))
    if cur:
        batches.append(cur)
    return batches


def run_batched(items, fn, durs, max_padded_sec=240.0, max_bs=32, cpu_fn=None, label=""):
    """fn(list of items) -> list of outputs, in item order. A CUDA OOM halves the batch; a single item goes to CPU."""
    out = [None] * len(items)
    t0, done_n = time.time(), 0

    def go(idx):
        try:
            res = fn([items[i] for i in idx])
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if len(idx) == 1:
                res = (cpu_fn or fn)([items[idx[0]]])
            else:
                h = len(idx) // 2
                go(idx[:h]); go(idx[h:])
                return
        for i, r in zip(idx, res):
            out[i] = r

    batches = pack_batches(durs, max_padded_sec, max_bs)
    for bi, b in enumerate(batches):
        go(b)
        done_n += len(b)
        if bi % 50 == 0 or done_n == len(items):
            el = time.time() - t0
            log(f"  {label} {done_n}/{len(items)}  {el:.0f}s  eta {el / done_n * (len(items) - done_n):.0f}s")
    return out


def cos_rows(a, b):
    a, b = np.asarray(a, np.float64), np.asarray(b, np.float64)
    return np.sum(a * b, 1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-12)


def amp(flag):
    return torch.autocast("cuda", dtype=torch.float16, enabled=bool(flag) and DEV.type == "cuda")


def read_rgb(src):
    """PIL RGB image; an image that cannot be read -> a black 224 x 224 image, listed in BROKEN["image"]."""
    from PIL import Image
    try:
        return Image.open(io.BytesIO(src) if isinstance(src, (bytes, bytearray)) else src).convert("RGB")
    except Exception as e:
        BROKEN["image"].append(f"{_src_name(src)} | {type(e).__name__}: {e}")
        log("!! unreadable image -> black image:", BROKEN["image"][-1])
        return Image.new("RGB", (224, 224))

## 3. Track 3 encoder helpers (FaRL ep16 download, CLIP-L id; same cell as FLAG_40)

In [ ]:
# ============================================================================ T3ENC-01 encoders: the encoder family of Track 3's VoxBlink system, frozen
# Track 3 (a teammate's VoxBlink pipeline, CB Bangla ng ~16 / g ~25) used these four encoders; here they run on OUR rows
# (MAV-Celeb v4 train, v1/v2, dev lists; and VoxCeleb2 in FLAG_41), so we can test locally whether the encoders and a
# cross-modal head carry the gain WITHOUT VoxBlink data (docs/PLAN_VOXBLINK_2026-10-07.md, V1). Every sample on its own.
#   farl16V   FaRL ViT-B/16 LAION-Face 20M **ep16** (github.com/FacePerceiver/FaRL release, 652 MB) on openai CLIP ViT-B/16
#             (CLIP preprocessing: 224 bicubic, centre crop); 512-d image embedding (our existing `farl` is the ep64 copy)
#   clipLV    openai/clip-vit-large-patch14 @ 32bd642 (transformers): get_image_features, 768-d
#   sam100pA  WeSpeaker voxblink2_samresnet100 (VoxBlink2 pretrained), official ModelScope zip (wenet/wespeaker_pretrained_models)
#   sam100fA  WeSpeaker voxblink2_samresnet100_ft (VoxBlink2 pretrained -> VoxCeleb2 large-margin fine-tune)
#             both: wespeaker fbank + CMN, int16-scaled waveform (as our wssim34 / wsr221 cells), 256-d
import zipfile as _zf

FARL16_URL = "https://github.com/FacePerceiver/FaRL/releases/download/pretrained_weights/FaRL-Base-Patch16-LAIONFace20M-ep16.pth"
FARL16_SIZE = 652165542
CLIPL_ID, CLIPL_REV = "openai/clip-vit-large-patch14", "32bd64288804d66eefd0ccbe215aa642df71cc41"
WESPK_TREE = "https://modelscope.cn/api/v1/datasets/wenet/wespeaker_pretrained_models/oss/tree"
SAM_ZIPS = {"sam100pA": ("voxblink2_samresnet100.zip", 186890339), "sam100fA": ("voxblink2_samresnet100_ft.zip", 186890839)}
T3_STATS = {}


def fetch(url, dest, size=None):
    dest = Path(dest)
    if dest.exists() and (size is None or dest.stat().st_size == size):
        return dest
    torch.hub.download_url_to_file(url, str(dest) + ".part", progress=False)
    Path(str(dest) + ".part").replace(dest)
    assert size is None or dest.stat().st_size == size, f"{dest.name}: {dest.stat().st_size} bytes, expected {size}"
    return dest


class FaRL16Face:
    kind = "face"

    def __init__(self):
        import clip
        m, self.pre = clip.load("ViT-B/16", device="cpu", download_root=str(CACHE / "clip"))
        sd = torch.load(fetch(FARL16_URL, CACHE / "FaRL-ep16.pth", FARL16_SIZE), map_location="cpu", weights_only=False)["state_dict"]
        vis = [k for k in m.state_dict() if k.startswith("visual.")]
        hit = [k for k in vis if k in sd and sd[k].shape == m.state_dict()[k].shape]
        assert len(hit) == len(vis), f"FaRL ep16 covers {len(hit)}/{len(vis)} visual tensors"
        m.load_state_dict(sd, strict=False)
        self.m = m.visual.float().to(DEV).eval()

    def prep(self, im):
        return self.pre(im)

    @torch.no_grad()
    def __call__(self, x, use_amp=True):
        return {"": self.m(x.to(DEV)).float().cpu().numpy()}


class CLIPLFace:
    kind = "face"

    def __init__(self):
        from transformers import CLIPModel, CLIPImageProcessor
        self.proc = CLIPImageProcessor.from_pretrained(CLIPL_ID, revision=CLIPL_REV)
        self.m = CLIPModel.from_pretrained(CLIPL_ID, revision=CLIPL_REV, torch_dtype=torch.float32).to(DEV).eval()

    def prep(self, im):
        return self.proc(images=im.convert("RGB"), return_tensors="pt").pixel_values[0]

    @torch.no_grad()
    def __call__(self, x, use_amp=True):
        f = self.m.get_image_features(pixel_values=x.to(DEV))
        f = f.pooler_output if hasattr(f, "pooler_output") else f          # transformers 5 may return an output object
        return {"": f.float().cpu().numpy()}


def wespeaker_dir(zip_name, size):
    """The official zip (ModelScope, as wespeaker's own hub does), unpacked without the __MACOSX entries."""
    d = CACHE / "wespeaker" / zip_name[:-4]
    if (d / "avg_model.pt").exists() and (d / "config.yaml").exists():
        return d
    d.mkdir(parents=True, exist_ok=True)
    import requests
    tree = requests.get(WESPK_TREE, timeout=120).json()["Data"]
    url = [it["Url"] for it in tree if it.get("Key") == zip_name][0]
    z = fetch(url, d / zip_name, size)
    with _zf.ZipFile(z) as zf:
        for mem in zf.namelist():
            if not mem.startswith("__MACOSX") and not mem.endswith("/"):
                (d / Path(mem).name).write_bytes(zf.read(mem))
    # some releases (voxceleb_resnet293_LM) name the pair <model>.pt / <model>.yaml instead of avg_model.pt / config.yaml
    pts, ymls = sorted(d.glob("*.pt")), sorted(d.glob("*.yaml"))
    if not (d / "avg_model.pt").exists() and len(pts) == 1:
        pts[0].rename(d / "avg_model.pt")
    if not (d / "config.yaml").exists() and len(ymls) == 1:
        ymls[0].rename(d / "config.yaml")
    assert (d / "avg_model.pt").exists() and (d / "config.yaml").exists(), f"{zip_name}: unexpected content {zf.namelist()[:6]}"
    return d


class SimAMVoice:
    kind = "voice"

    def __init__(self, name):
        for mod, attr in (("wespeaker.frontend.s3prl", "S3prlFrontend"), ("wespeaker.frontend.whisper_encoder", "whisper_encoder"),
                          ("wespeaker.frontend.w2vbert", "W2VBertFrontend"), ("wespeaker.diar.umap_clusterer", "cluster"),
                          ("wespeaker.diar.extract_emb", "subsegment")):
            if mod not in sys.modules:                     # SSL frontends / diarisation helpers: not used by fbank models
                stub = types.ModuleType(mod); setattr(stub, attr, None); sys.modules[mod] = stub
        import wespeaker
        self.m = wespeaker.load_model(str(wespeaker_dir(*SAM_ZIPS[name])))
        self.m.set_device("cuda:0" if DEV.type == "cuda" else "cpu")
        T3_STATS[name] = {"params_M": round(sum(p.numel() for p in self.m.model.parameters()) / 1e6, 1)}

    @torch.no_grad()
    def __call__(self, w, use_amp=True):
        pcm = torch.from_numpy(np.ascontiguousarray(w, dtype=np.float32) * 32768.0)[None]
        feats = self.m.compute_features(pcm, sample_rate=16000, cmn=True)
        out = self.m.model(feats.to(self.m.device))
        out = out[-1] if isinstance(out, tuple) else out
        return {"": out[0].reshape(-1).float().cpu().numpy()}


NEW = {"farl16V": FaRL16Face, "clipLV": CLIPLFace, "sam100pA": lambda: SimAMVoice("sam100pA"),
       "sam100fA": lambda: SimAMVoice("sam100fA")}


def build(name):
    return NEW[name]()


def release(keep_ib=False):
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


ARRAYS_NEW = {"farl16V": ("face", {"": "farl16"}), "clipLV": ("face", {"": "clipL"}),
              "sam100pA": ("voice", {"": "sam100p"}), "sam100fA": ("voice", {"": "sam100f"})}
MAIN_NEW = {k: "" for k in NEW}
RESHAPE = {}

## 4. FACELORA-02: frames, wiring check, LoRA, training (registered settings)

In [ ]:
# ============================================================================ FACELORA-02 adapter training (registered settings; no selection by result)
import math, tarfile, requests, struct, zlib, threading
import concurrent.futures as cf
from huggingface_hub import HfFileSystem
from torch.utils.checkpoint import checkpoint

SMOKE = (LOCAL or os.environ.get("FLAG_SMOKE") == "1")
CFG = dict(enc=LORA_ENC, rank=16, alpha=32, blocks={"farl16": list(range(6, 12)), "clipL": list(range(18, 24))}[LORA_ENC],
           P=int(os.environ.get("LORA_P", "32")), K=int(os.environ.get("LORA_K", "4")), steps=int(os.environ.get("LORA_STEPS", "1500")),
           tau=0.05, lr_lora=1e-4, lr_proj=1e-3, wd=1e-2, warmup=100, held_out=300, seed=0, flip=0.5, grad_ckpt=True)
SMOKE_IDS = int(os.environ.get("VOX_SMOKE_IDS", "0")) if SMOKE else 0
REF_FACE = {"farl16": "face_farl16.npy", "clipL": "face_clipL.npy"}   # FLAG_41 frozen arrays (wiring check)
ZIPS = [2, 3, 4, 5, 6]
ZIP_PATH = "datasets/ProgramComputer/voxceleb/vox2/vox2_mp4_{}.zip"
ZIP_URL = "https://huggingface.co/datasets/ProgramComputer/voxceleb/resolve/main/vox2/vox2_mp4_{}.zip"
FS = HfFileSystem()
FRAMES = OUT / "vox_frames_png.zip"                  # lossless: CLIP-L moves to cos ~0.93 under JPEG q92
FILES = SCAN["files"]                                                           # the data cell's one walk over the inputs
for n in ("vox_frames_png.zip", "facelora_state.pt", "train_log.json"):          # resume: cached frames, adapter
    prev = [p for p in FILES.get(n, []) if p.parent.name == OUT.name]
    if prev and not (OUT / n).exists():
        shutil.copy(prev[0], OUT / n); log("resume:", n, "from", prev[0].parent)
if not FRAMES.exists() and not SMOKE:                                            # frames cached by the other FACELORA notebook
    fz = sorted(FILES.get("vox_frames_png.zip", []), key=lambda f: -f.stat().st_size)
    if fz:
        shutil.copy(fz[0], FRAMES); log("frames reused from", fz[0])

# ---- the VoxCeleb2 samples of FLAG_37 / 41: key lists + frozen SimAM voice embeddings + the frozen face arrays (wiring check)
# FLAG_41's arrays (its output, or the copy in flag2027-lab-data: lab/Experiment/T3ENC-01/feats_vox2t3) and FLAG_37's key
# lists (beside them in FLAG_41's output, or lab/Experiment/VOX2-01/feats_vox2; the same files)
vt3 = [p for p in FILES.get("voice_sam100p.npy", []) if "feats_facelora" not in p.as_posix()]
assert vt3, "attach flag2027-lab-data (or the output of FLAG_41): VoxCeleb2 T3 arrays + key lists"
V41 = vt3[0].parent
fmeta = [p for p in FILES.get("face_meta.csv", []) if (p.parent / "voice_meta.csv").exists() and "feats_facelora" not in p.as_posix()]
META = V41 if (V41 / "face_meta.csv").exists() else sorted(fmeta, key=lambda p: "feats_vox2" not in p.as_posix())[0].parent
for f in V41.glob("vox2_mp4_*_listing.csv"):                                     # FLAG_41's zip listings (if attached): no HF listing
    if not (OUT / f.name).exists():
        shutil.copy(f, OUT / f.name)
FK = pd.read_csv(META / "face_meta.csv", dtype={"spk": str, "video": str, "utt": str})
VK = pd.read_csv(META / "voice_meta.csv", dtype={"spk": str, "video": str, "utt": str})
log(f"VoxCeleb2 arrays from {V41}, key lists from {META}")
FK["row0"] = np.arange(len(FK))
l2n_ = lambda X: X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
VE = np.concatenate([l2n_(np.load(V41 / f"voice_{n}.npy").astype(np.float32)) for n in ("sam100p", "sam100f")], 1)
assert len(VE) == len(VK) and np.isfinite(VE).all() and len(np.load(V41 / REF_FACE[LORA_ENC], mmap_mode="r")) == len(FK)
lid = [p for p in FILES.get("lid_summary.json", []) if "vox2" in p.as_posix().lower() or "nb-42" in p.as_posix()] or FILES.get("lid_summary.json", [])
assert lid, "attach flag2027-lab-data (feats_vox2lid) or the output of FLAG_42: lid_summary.json, the Bengali filter is REQUIRED"
DROP = set(json.loads(lid[0].read_text())["speakers_dropped_ids"])
FILTERED = True
log(f"VoxCeleb2: {len(FK)} faces, {len(VK)} voices, {FK.spk.nunique()} ids | Bengali filter (FLAG_42): {len(DROP)} speakers dropped")
if SMOKE_IDS:
    keep = sorted((set(FK.spk) & set(VK.spk)) - DROP)[:SMOKE_IDS]
    fk_m, vk_m = FK.spk.isin(keep).values, VK.spk.isin(keep).values
    FK, VK, VE = FK[fk_m].reset_index(drop=True), VK[vk_m].reset_index(drop=True), VE[vk_m]
    CFG["held_out"] = 1
    log(f"smoke test: {len(keep)} VoxCeleb2 identities, {len(FK)} faces, {len(VK)} voices")


# ---- frames: the middle frame of every FLAG_37 face track, cached as lossless PNG in vox_frames_png.zip (resumable per chunk);
#      FLAG_41 embedded the decoded frame itself, so the step-0 wiring check compares the same pixels
def get_retry(url, tries=4, **kw):
    for k in range(tries):
        try:
            r = requests.get(url, timeout=300, **kw); r.raise_for_status(); return r
        except Exception as e:
            log(f"  retry {k + 1}/{tries}: {type(e).__name__}: {e}"); time.sleep(10 * (k + 1))
    raise RuntimeError(f"cannot fetch {url}")


def zip_listing(k):
    f = OUT / f"vox2_mp4_{k}_listing.csv"
    if f.exists():
        return pd.read_csv(f, dtype={"video": str, "utt": str})
    with FS.open(ZIP_PATH.format(k), "rb", block_size=2 ** 19) as fh:
        rows = [(i.filename, i.header_offset, i.compress_size, i.compress_type)
                for i in zipfile.ZipFile(fh).infolist() if i.filename.endswith(".mp4")]
    L_ = pd.DataFrame(rows, columns=["name", "off", "csize", "ctype"])
    sp = L_.name.str.split("/")
    L_["spk"], L_["video"], L_["utt"], L_["zip"] = sp.str[-3], sp.str[-2], sp.str[-1].str[:-4], k
    L_.to_csv(f, index=False)
    return L_


_TL, URLS = threading.local(), {}


def resolve(k):
    with requests.get(ZIP_URL.format(k), headers={"Range": "bytes=0-0"}, allow_redirects=True, timeout=120, stream=True) as r:
        return r.url


def middle_frame_png(mp4):
    import av
    with av.open(io.BytesIO(mp4)) as c:
        st = c.streams.video[0]; n = st.frames or 50
        last = None
        for k, fr in enumerate(c.decode(st)):
            last = fr
            if k >= n // 2:
                break
        if last is None:
            return None
        buf = io.BytesIO(); last.to_image().convert("RGB").save(buf, "PNG", compress_level=3); return buf.getvalue()


def fetch_jpg(job):
    k, off, csize, ctype = job
    if not hasattr(_TL, "s"):
        _TL.s = requests.Session()
    for attempt in range(4):
        try:
            r = _TL.s.get(URLS[k], headers={"Range": f"bytes={off}-{off + 30 + 1024 + csize}"}, timeout=120)
            if r.status_code in (401, 403):
                URLS[k] = resolve(k); continue
            b = r.content
            n, e = struct.unpack("<HH", b[26:30])
            d = b[30 + n + e:30 + n + e + csize]
            return middle_frame_png(zlib.decompress(d, -15) if ctype == 8 else d)
        except Exception:
            time.sleep(2 * (attempt + 1))
    return None


have = set()
if FRAMES.exists():
    with zipfile.ZipFile(FRAMES) as z:
        have = {int(n[:-4]) for n in z.namelist() if n.endswith(".png")}
todo = [int(r) for r in FK.row0 if int(r) not in have]                          # frame names = FLAG_37 row numbers
if todo:
    LST = pd.concat([zip_listing(k) for k in ZIPS], ignore_index=True)
    for k in ZIPS:
        URLS[k] = resolve(k)
    SEL = FK.set_index("row0").loc[todo, ["spk", "video", "utt"]].reset_index().merge(
        LST[["spk", "video", "utt", "zip", "off", "csize", "ctype"]], how="left", on=["spk", "video", "utt"])
    assert SEL.zip.notna().all(), f"{SEL.zip.isna().sum()} face keys not in the zip listings"
    CH = 2400
    log(f"frames: {len(SEL)} to fetch ({len(have)} cached), {len(SEL) // CH + 1} chunks")
    for c0 in range(0, len(SEL), CH):
        t0 = time.time(); S_ = SEL.iloc[c0:c0 + CH]
        with cf.ThreadPoolExecutor(16) as ex:
            jpgs = list(ex.map(fetch_jpg, zip(S_.zip.astype(int), S_.off.astype(int), S_.csize.astype(int), S_.ctype.astype(int))))
        with zipfile.ZipFile(FRAMES, "a", zipfile.ZIP_STORED) as z:
            for row, b in zip(S_.row0, jpgs):
                if b is not None:
                    z.writestr(f"{row}.png", b)
        n_ok = sum(b is not None for b in jpgs)
        log(f"  chunk {c0 // CH}: {n_ok}/{len(S_)} frames [{time.time() - t0:.0f}s]")
wanted = set(int(r) for r in FK.row0)
with zipfile.ZipFile(FRAMES) as z:
    JPG0 = {int(n[:-4]): z.read(n) for n in z.namelist() if n.endswith(".png") and int(n[:-4]) in wanted}
JPG = {i: JPG0[int(r)] for i, r in enumerate(FK.row0) if int(r) in JPG0}          # by row of FK (smoke: re-indexed)
log(f"frames in memory: {len(JPG)}/{len(FK)} ({sum(len(b) for b in JPG.values()) / 1e6:.0f} MB)")

# ---- the face encoder + LoRA, voice projection
if LORA_ENC == "farl16":
    import clip as _clip
    _base, PRE = _clip.load("ViT-B/16", device="cpu", download_root=str(CACHE / "clip"))
    _sd = torch.load(fetch(FARL16_URL, CACHE / "FaRL-ep16.pth", FARL16_SIZE), map_location="cpu", weights_only=False)["state_dict"]
    _vis = [k for k in _base.state_dict() if k.startswith("visual.")]
    assert all(k in _sd and _sd[k].shape == _base.state_dict()[k].shape for k in _vis)
    _base.load_state_dict(_sd, strict=False)
    VIS = _base.visual.float()
    LAYERS, MLP_NAMES = VIS.transformer.resblocks, ("c_fc", "c_proj")
    ENC_FWD = lambda x: VIS(x)
    DIM = int(VIS.proj.shape[1])
else:
    from transformers import CLIPModel, CLIPImageProcessor
    _proc = CLIPImageProcessor.from_pretrained(CLIPL_ID, revision=CLIPL_REV)
    _cm = CLIPModel.from_pretrained(CLIPL_ID, revision=CLIPL_REV, torch_dtype=torch.float32)
    VIS = nn.ModuleDict(dict(vm=_cm.vision_model, proj=_cm.visual_projection)).float()
    del _cm
    PRE = lambda im: _proc(images=im.convert("RGB"), return_tensors="pt").pixel_values[0]
    LAYERS, MLP_NAMES = VIS["vm"].encoder.layers, ("fc1", "fc2")
    ENC_FWD = lambda x: VIS["proj"](VIS["vm"](pixel_values=x).pooler_output)     # = get_image_features
    DIM = int(VIS["proj"].out_features)
assert DIM == {"farl16": 512, "clipL": 768}[LORA_ENC]
REF = REF_FACE[LORA_ENC]
for p in VIS.parameters():
    p.requires_grad = False


class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + F.linear(x, self.B @ self.A) * self.scale


class Ckpt(nn.Module):
    """Gradient checkpointing of one adapted block during training (memory only; the same function)."""
    def __init__(self, blk):
        super().__init__(); self.blk = blk

    def forward(self, *a, **k):
        if self.training and torch.is_grad_enabled():
            return checkpoint(self.blk, *a, use_reentrant=False, **k)
        return self.blk(*a, **k)


def add_lora(cfg):
    names = []
    for i in cfg["blocks"]:
        mlp = LAYERS[i].mlp
        for n in MLP_NAMES:
            setattr(mlp, n, LoRALinear(getattr(mlp, n), cfg["rank"], cfg["alpha"])); names.append(f"layer{i}.mlp.{n}")
    if cfg["grad_ckpt"]:
        for i in cfg["blocks"]:
            LAYERS[i] = Ckpt(LAYERS[i])
    return names


LORA_NAMES = add_lora(CFG)
VIS = VIS.to(DEV)
VPROJ = nn.Linear(VE.shape[1], DIM).to(DEV)
with torch.no_grad():
    VPROJ.bias.zero_()
lora_params = [p for n, p in VIS.named_parameters() if p.requires_grad]
log(f"{LORA_ENC} LoRA: {len(LORA_NAMES)} layers, {sum(p.numel() for p in lora_params) / 1e6:.2f} M trainable "
    f"(+ vproj {sum(p.numel() for p in VPROJ.parameters()) / 1e6:.2f} M), output {DIM}")

# ---- identities, held-out, batch sampler
from PIL import Image
rng = np.random.RandomState(CFG["seed"]); torch.manual_seed(CFG["seed"])
fv = FK.groupby("spk").indices; vv = VK.groupby("spk").indices
ids = sorted(s for s in set(fv) & set(vv) if s not in DROP and any(i in JPG for i in fv[s]))
held = sorted(rng.choice(ids, min(CFG["held_out"], max(1, len(ids) // 10)), replace=False)) if len(ids) > 2 else []
train_ids = [s for s in ids if s not in set(held)]
fvid, vvid = FK.video.values, VK.video.values
GEN = dict(zip(FK.spk, FK.gender)) if "gender" in FK else {}
log(f"identities: {len(ids)} usable ({len(DROP)} dropped) -> train {len(train_ids)}, held out {len(held)}")


def prep_jpg(i, flip):
    im = Image.open(io.BytesIO(JPG[i])).convert("RGB")
    if flip:
        im = im.transpose(Image.FLIP_LEFT_RIGHT)
    return PRE(im)


def sample_batch():
    P, K = CFG["P"], CFG["K"]
    spk = rng.choice(train_ids, min(P, len(train_ids)), replace=False)
    fi, vi, lab = [], [], []
    for j, s in enumerate(spk):
        faces = [i for i in fv[s] if i in JPG]
        for _ in range(K):
            v = rng.choice(vv[s])
            cand = [i for i in faces if fvid[i] != vvid[v]] or faces
            fi.append(int(rng.choice(cand))); vi.append(int(v)); lab.append(j)
    flips = rng.rand(len(fi)) < CFG["flip"]
    with cf.ThreadPoolExecutor(8) as ex:
        X = torch.stack(list(ex.map(prep_jpg, fi, flips)))
    return X, torch.from_numpy(VE[vi]), torch.tensor(lab)


def multi_pos_infonce(S, lab):
    """S: logits (n x n), lab: identity of every row; positives = same identity (K per row); symmetric over rows and columns."""
    same = (lab[:, None] == lab[None, :]).float()
    def one(L):
        logp = L - torch.logsumexp(L, 1, keepdim=True)
        return -((logp * same).sum(1) / same.sum(1)).mean()
    return 0.5 * (one(S) + one(S.t()))


@torch.no_grad()
def embed_faces(rows, bs=128):
    VIS.eval(); out = []
    for s in range(0, len(rows), bs):
        X = torch.stack([prep_jpg(i, False) for i in rows[s:s + bs]]).to(DEV)
        with torch.autocast("cuda", enabled=DEV.type == "cuda"):
            out.append(F.normalize(ENC_FWD(X).float(), dim=1).cpu())
    return torch.cat(out).numpy()


def sanity_eer():
    """Held-out identities: face of one video vs voice of ANOTHER video; negatives = other held-out identities (all / same gender)."""
    from sklearn.metrics import roc_curve
    if not held:
        return {}
    rows = [i for s in held for i in fv[s] if i in JPG]
    Fh = dict(zip(rows, embed_faces(rows)))
    VPROJ.eval()
    with torch.no_grad():
        Vh = F.normalize(VPROJ(torch.from_numpy(VE).to(DEV)).float(), dim=1).cpu().numpy()
    r2 = np.random.RandomState(1); pos, neg, negg = [], [], []
    for s in held:
        fr = [i for i in fv[s] if i in JPG]
        for v in vv[s][:4]:
            cand = [i for i in fr if fvid[i] != vvid[v]]
            if cand:
                pos.append(float(Fh[r2.choice(cand)] @ Vh[v]))
            o = held[r2.randint(len(held))]
            if o != s:
                fo = [i for i in fv[o] if i in JPG]
                sc = float(Fh[r2.choice(fo)] @ Vh[v]); neg.append(sc)
                if GEN.get(o) == GEN.get(s):
                    negg.append(sc)
    def eer(p, n):
        if len(p) < 2 or len(n) < 2:
            return None
        y = np.r_[np.ones(len(p)), np.zeros(len(n))]; s_ = np.r_[p, n]
        fpr, tpr, _ = roc_curve(y, s_); fnr = 1 - tpr; i = np.nanargmin(np.abs(fnr - fpr)); return round(100 * float(fpr[i] + fnr[i]) / 2, 2)
    return {"ng": eer(pos, neg), "g": eer(pos, negg) if len(negg) > 20 else None, "pos": len(pos), "neg": len(neg)}


# ---- train (or load)
STATE = OUT / "facelora_state.pt"
if STATE.exists():
    st = torch.load(STATE, map_location="cpu", weights_only=False)
    assert st["cfg"]["enc"] == LORA_ENC, st["cfg"]
    VIS.load_state_dict({**VIS.state_dict(), **{k: v for k, v in st["vis"].items()}}); VPROJ.load_state_dict(st["vproj"])
    log("adapter loaded from the previous output:", st["cfg"])
    WIRING = st.get("wiring")
else:
    # wiring check: at step 0 (B = 0) the adapter is the frozen encoder -> FLAG_41's frozen arrays on the same frames
    chk = [i for i in range(len(FK)) if i in JPG][:64]
    ref = l2n_(np.load(V41 / REF, mmap_mode="r")[FK.row0.values[chk]].astype(np.float32))
    cos = (embed_faces(chk) * ref).sum(1)
    WIRING = dict(rows=len(chk), mean_cos=round(float(cos.mean()), 5), min_cos=round(float(cos.min()), 5))
    log(f"wiring check vs FLAG_41 {REF} at step 0: {WIRING}")
    assert WIRING["mean_cos"] >= 0.99, f"the adapter at step 0 does not reproduce the frozen {LORA_ENC} arrays: {WIRING}"
    opt = torch.optim.AdamW([{"params": lora_params, "lr": CFG["lr_lora"]}, {"params": VPROJ.parameters(), "lr": CFG["lr_proj"]}], weight_decay=CFG["wd"])
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / CFG["warmup"]) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / max(1, CFG["steps"])))))
    scaler = torch.amp.GradScaler(enabled=DEV.type == "cuda")
    LOG = {"cfg": CFG, "filtered": FILTERED, "wiring": WIRING, "loss": [], "sanity": {}}
    LOG["sanity"]["0"] = sanity_eer(); log("sanity EER before training:", LOG["sanity"]["0"])
    t0 = time.time()
    for step in range(1, CFG["steps"] + 1):
        VIS.train(); VPROJ.train()
        X, V_, lab = sample_batch()
        X, V_, lab = X.to(DEV), V_.to(DEV), lab.to(DEV)
        with torch.autocast("cuda", enabled=DEV.type == "cuda"):
            f = F.normalize(ENC_FWD(X).float(), dim=1); v = F.normalize(VPROJ(V_).float(), dim=1)
            loss = multi_pos_infonce(f @ v.t() / CFG["tau"], lab)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        LOG["loss"].append(float(loss))
        if step % 50 == 0 or step == CFG["steps"]:
            log(f"step {step}/{CFG['steps']} loss {np.mean(LOG['loss'][-50:]):.4f} [{time.time() - t0:.0f}s]"
                + (f" | GPU peak {torch.cuda.max_memory_allocated() / 2 ** 30:.1f} GB" if DEV.type == "cuda" else ""))
        if step in (500, 1000, CFG["steps"]):
            LOG["sanity"][str(step)] = sanity_eer(); log(f"sanity EER at step {step}:", LOG["sanity"][str(step)])
    torch.save({"vis": {k: v.cpu() for k, v in VIS.state_dict().items() if k.endswith((".A", ".B"))}, "vproj": VPROJ.state_dict(),
                "cfg": CFG, "lora_names": LORA_NAMES, "filtered": FILTERED, "dropped": sorted(DROP), "wiring": WIRING}, STATE)
    (OUT / "train_log.json").write_text(json.dumps(LOG, indent=1))
    log("adapter saved ->", STATE.name)
VIS.eval(); VPROJ.eval()
np.save(OUT / "vproj_W.npy", VPROJ.weight.detach().cpu().numpy()); np.save(OUT / "vproj_b.npy", VPROJ.bias.detach().cpu().numpy())

## 5. The adapted encoder

In [ ]:
# ============================================================================ the adapted encoder, in the FLAG_12 encoder interface
class Adapted:
    kind = "face"

    def __init__(self):
        VIS.eval()

    def prep(self, im):
        return PRE(im)

    @torch.no_grad()
    def __call__(self, x, use_amp=True):
        with torch.autocast("cuda", enabled=use_amp and DEV.type == "cuda"):
            return {"": ENC_FWD(x.to(DEV)).float().cpu().numpy()}


NEW = {ARR + "V": Adapted}


def build(name):
    return NEW[name]()


def release(keep_ib=False):
    import gc
    gc.collect()


ARRAYS_NEW = {ARR + "V": ("face", {"": ARR})}
MAIN_NEW = {k: "" for k in NEW}
RESHAPE = {}

## 6. `farlA` of every MAV-Celeb row (FLAG_12 loop, resumable)

In [ ]:
# ============================================================================ running
def run_faces_tensor(enc, srcs, bs=64, label=""):
    out, t0 = {}, time.time()
    for s in range(0, len(srcs), bs):
        x = torch.stack([enc.prep(read_rgb(p() if callable(p) else p)) for p in srcs[s:s + bs]])
        for k, v in enc(x).items():
            out.setdefault(k, []).append(v)
        if (s // bs) % 40 == 0:
            log(f"  {label} {s + len(x)}/{len(srcs)}  {time.time() - t0:.0f}s")
    return {k: np.concatenate(v).astype(np.float32) for k, v in out.items()}


def run_voices_single(enc, n, load, label=""):
    """One file at a time: no padding enters a mean."""
    out, t0 = {}, time.time()
    for i in range(n):
        for k, v in enc(load(i)).items():
            out.setdefault(k, []).append(np.atleast_1d(v))
        if i % 1000 == 0 or i == n - 1:
            el = time.time() - t0
            log(f"  {label} {i + 1}/{n}  {el:.0f}s  eta {el / (i + 1) * (n - i - 1):.0f}s")
    return {k: np.stack(v).astype(np.float32) for k, v in out.items()}


# output names: {kind}_{array}_{split}.npy for organiser splits, ext_{src}_{kind}_{array}.npy for v1/v2
ARRAYS = {"arcface": ("face", {"": "arcface", "_detok": "arcface_detok"}),
          "ecapa": ("voice", {"192": "ecapa192", "6144": "ecapa6144"}),
          "vitage": ("face", {"": "vitage", "prob": "vitageprob"}),
          "w2v2ag": ("voice", {"": "w2v2ag", "age": "w2v2agage", "gen": "w2v2aggen"}),
          "ibv": ("face", {"": "ibv"}), "iba": ("voice", {"": "iba"})}
MAIN = {"arcface": "", "ecapa": "192", "vitage": "", "w2v2ag": "", "ibv": "", "iba": ""}
ARRAYS.update(ARRAYS_NEW); MAIN.update(MAIN_NEW)                         # this notebook's arrays


def _dt(v, split):
    """float16 for the v1/v2 rows (as FLAG_12) and for every wide array (layers / tokens); float32 otherwise."""
    return v if (split and v.ndim == 2 and v.shape[1] <= 4096) or v.ndim == 1 else v.astype(np.float16)


def target(name, suffix, split=None, src=None):
    kind, names = ARRAYS[name]
    return OUT / (f"{kind}_{names[suffix]}_{tag(split)}.npy" if split else f"ext_{src}_{kind}_{names[suffix]}.npy")


def complete(name, split=None, src=None):
    return all(target(name, s, split, src).exists() for s in ARRAYS[name][1])


def save(name, R, split=None, src=None):
    for s, v in R.items():                         # main array last: its presence marks the job as done
        if s != MAIN[name]:
            np.save(target(name, s, split, src), _dt(v, split))
    np.save(target(name, MAIN[name], split, src), _dt(R[MAIN[name]], split))


def extract(name, enc, split=None, src=None):
    kind = ARRAYS[name][0]
    if split:
        sp = S[split]
        faces = [sp["root"] / p for p in sp["jpg"]]
        load = lambda i: load_audio(sp["root"] / sp["wav"][i])
        n, durs, label = len(sp["wav"]), sp["dur"], split
    else:
        rows = EXT[src]
        faces = [rows.read(p) for p in rows.face] if name == "arcface" else [(lambda p=p: rows.read(p)) for p in rows.face]
        load = lambda i: centre_crop(load_audio(io.BytesIO(rows.read(rows.voice[i]))), rows.L)
        n, label = len(rows.voice), src
        durs = np.full(n, 12.0, np.float32)          # upper bound after the crop: batches stay within budget
    if kind == "face":
        R = enc.embed(faces, label=label) if name == "arcface" else run_faces_tensor(enc, faces, label=f"{name} {label}")
    elif name == "ecapa":
        R = enc.embed(load, n, durs, label=label)
    else:
        R = run_voices_single(enc, n, load, label=f"{name} {label}")
    save(name, R, split, src)
    log(f"saved {target(name, MAIN[name], split, src).name} {R[MAIN[name]].shape}")


def self_check(name, enc):
    """Batching / fp16 must not change the embeddings (min cosine over a few rows; must be > 0.995)."""
    k = next(k for k in S if k != "train") if len(S) > 1 else next(iter(S))
    sp = S[k]
    if name == "arcface":
        return {}
    if name == "ecapa":
        idx = list(np.argsort(-sp["dur"])[:3]) + list(range(min(5, len(sp["wav"]))))
        sigs = [load_audio(sp["root"] / sp["wav"][i]) for i in idx]
        b, s = enc.batch(sigs), [enc.batch([x])[0] for x in sigs]
        return {"batch192": float(cos_rows([r[0] for r in b], [r[0] for r in s]).min()),
                "batch6144": float(cos_rows([r[1] for r in b], [r[1] for r in s]).min())}
    if ARRAYS[name][0] == "face":
        x = torch.stack([enc.prep(read_rgb(sp["root"] / p)) for p in sp["jpg"][:6]])
        full = enc(x, use_amp=False)[""]
        single = np.concatenate([enc(x[i:i + 1], use_amp=False)[""] for i in range(len(x))])
        return {"batch": float(cos_rows(full, single).min()), "fp16": float(cos_rows(full, enc(x)[""]).min())}
    waves = [load_audio(sp["root"] / p) for p in sp["wav"][:3]]
    return {"fp16": float(cos_rows([enc(w, use_amp=False)[""] for w in waves], [enc(w)[""] for w in waves]).min())}


CHECKS, FAILED = {}, {}
for name in ENCODERS:
    jobs = [("split", k) for k in S if not complete(name, split=k)]
    if name in EXT_ENCODERS:
        jobs += [("src", s) for s in EXT if not complete(name, src=s)]
    if not jobs:
        log(f"== {name}: every array already exists -> skipped")
        continue
    t0 = time.time()
    try:
        enc = build(name)
        CHECKS[name] = self_check(name, enc)
        log(name, "self-check:", CHECKS[name])
        assert not CHECKS[name] or min(CHECKS[name].values()) > 0.995, f"{name}: batching / fp16 changes the output"
        for what, key in jobs:
            try:
                extract(name, enc, **({"split": key} if what == "split" else {"src": key}))
            except Exception as e:                  # one bad source must not cost the other splits
                FAILED[f"{name}:{key}"] = f"{type(e).__name__}: {e}"
                log(f"!! {name} on {key} FAILED: {FAILED[f'{name}:{key}']}")
    except Exception as e:
        FAILED[name] = f"{type(e).__name__}: {e}"
        log(f"!! {name} FAILED -> skipped: {FAILED[name]}")
    enc = None
    release(keep_ib=(name in ('ibv',)))
    log(f"== {name} done in {time.time() - t0:.0f}s")
print("failed:", FAILED or "none")

## 7. `farlA` of the VoxCeleb2 frames

In [ ]:
# ============================================================================ the adapted embedding of the VoxCeleb2 frames (FLAG_37 / 41 row order) + manifest extras
rows = [i for i in range(len(FK)) if i in JPG]
A = np.full((len(FK), DIM), np.nan, np.float16)
if rows:
    A[rows] = embed_faces(rows).astype(np.float16)
np.save(OUT / f"vox_face_{ARR}.npy", A); FK.drop(columns="row0").to_csv(OUT / "face_meta.csv", index=False)
TL = json.loads((OUT / "train_log.json").read_text()) if (OUT / "train_log.json").exists() else None
CHECKS["facelora"] = {"enc": LORA_ENC, "array": ARR, "filtered": FILTERED, "dropped_speakers": len(DROP), "train_ids": len(train_ids),
                      "held_out": len(held), "vox_frames_embedded": len(rows), "vox_rows": len(FK), "cfg": CFG, "wiring": WIRING,
                      "train_log": dict(TL, loss=TL["loss"][::50]) if TL else None}
log(f"VoxCeleb2 frames embedded: {len(rows)}/{len(FK)} | sanity EER:", TL["sanity"] if TL else "n/a")

## 8. Manifest

In [ ]:
man = {"created": time.strftime("%Y-%m-%d %H:%M"), "splits": {k: len(v["wav"]) for k, v in S.items()},
       "external": {k: [len(r.face), len(r.voice)] for k, r in EXT.items()}, "encoders": ENCODERS,
       "checks": CHECKS, "failed": FAILED, "audio": AUDIO_STATS, "broken": BROKEN, "resumed_arrays": len(SEEDED),
       "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
       "files": {f.name: list(np.load(f, mmap_mode="r").shape) for f in sorted(OUT.glob("*.npy"))}}
try:
    import speechbrain, insightface
    man["versions"].update(speechbrain=speechbrain.__version__, insightface=insightface.__version__)
except Exception:
    pass
man["reshape"] = RESHAPE                 # flattened wide arrays: row -> (L, D)
(OUT / "feats_manifest.json").write_text(json.dumps(man, indent=1))
shutil.rmtree(WORKDIR / "raw", ignore_errors=True)
tot = sum(p.stat().st_size for p in OUT.glob("*")) / 1e9
print(f"{len(man['files'])} arrays, {tot:.2f} GB in {OUT} | failed: {FAILED or 'none'}")
print("NEXT: Save Version (Save & Run All) -> Output -> New Dataset -> name it flag2027-feats-facelora (re-run with this output attached until every encoder is done)")